# Device Control AI Tutorial

This tutorial demonstrates how to use the Device Control AI component to manage and control quantum devices in a distributed environment.

In [ ]:
# Install required packages
!pip install paho-mqtt numpy matplotlib

In [ ]:
import json
import time
import paho.mqtt.client as mqtt
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Callable
from enum import Enum
import numpy as np
import matplotlib.pyplot as plt

## 1. Device Control Architecture

In [ ]:
class DeviceType(Enum):
    QUANTUM_PROCESSOR = "quantum_processor"
    CRYOGENIC_SYSTEM = "cryogenic_system"
    CONTROL_ELECTRONICS = "control_electronics"
    CLASSICAL_COMPUTER = "classical_computer"
    SENSOR = "sensor"

@dataclass
class DeviceState:
    device_id: str
    device_type: DeviceType
    status: str = "offline"
    parameters: Optional[Dict] = None
    last_updated: float = 0.0
    
    def update(self, new_state: Dict):
        """Update device state with new values."""
        for key, value in new_state.items():
            if hasattr(self, key):
                setattr(self, key, value)
        self.last_updated = time.time()
    
    def to_dict(self):
        """Convert device state to dictionary."""
        state = asdict(self)
        # Convert Enum to string
        state['device_type'] = self.device_type.value
        return state

## 2. MQTT Communication Layer

In [ ]:
class MQTTClient:
    def __init__(self, client_id: str, broker: str = \"localhost\", port: int = 1883):
    self.client = mqtt.Client(client_id=client_id)
    self.broker = broker\n",
    self.port = port\n",
    self.connected = False\n",
    "        self.subscriptions = {}\n",
    "        \n",
    "        # Set up callbacks\n",
    "        self.client.on_connect = self._on_connect\n",
    "        self.client.on_message = self._on_message\n",
    "        self.client.on_disconnect = self._on_disconnect\n",
    "    \n",
    "    def connect(self):\n",
    "        \"\"\"Connect to the MQTT broker.\"\"\"\n",
    "        try:\n",
    "            self.client.connect(self.broker, self.port, 60)\n",
    "            self.client.loop_start()\n",
    "            self.connected = True\n",
    "            print(f\"Connected to MQTT broker at {self.broker}:{self.port}\")\n",
    "        except Exception as e:\n",
    "            print(f\"Failed to connect to MQTT broker: {e}\")\n",
    "    \n",
    "    def disconnect(self):\n",
    "        \"\"\"Disconnect from the MQTT broker.\"\"\"\n",
    "        if self.connected:\n",
    "            self.client.loop_stop()\n",
    "            self.client.disconnect()\n",
    "            self.connected = False\n",
    "            print(\"Disconnected from MQTT broker\")\n",
    "    \n",
    "    def publish(self, topic: str, payload: Dict):\n",
    "        \"\"\"Publish a message to a topic.\"\"\"\n",
    "        if not self.connected:\n",
    "            print(\"Not connected to MQTT broker\")\n",
    "            return False\n",
    "        \n",
    "        try:\n",
    "            result = self.client.publish(\n",
    "                topic,\n",
    "                payload=json.dumps(payload),\n",
    "                qos=1,\n",
    "                retain=True\n",
    "            )\n",
    "            return result.rc == mqtt.MQTT_ERR_SUCCESS\n",
    "        except Exception as e:\n",
    "            print(f\"Failed to publish message: {e}\")\n",
    "            return False\n",
    "    \n",
    "    def subscribe(self, topic: str, callback: Callable):\n",
    "        \"\"\"Subscribe to a topic with a callback function.\"\"\"\n",
    "        if not self.connected:\n",
    "            print(\"Not connected to MQTT broker\")\n",
    "            return False\n",
    "        \n",
    "        try:\n",
    "            self.client.subscribe(topic, qos=1)\n",
    "            self.subscriptions[topic] = callback\n",
    "            print(f\"Subscribed to {topic}\")\n",
    "            return True\n",
    "        except Exception as e:\n",
    "            print(f\"Failed to subscribe to {topic}: {e}\")\n",
    "            return False\n",
    "    \n",
    "    def _on_connect(self, client, userdata, flags, rc):\n",
    "        if rc == 0:\n",
    "            print(\"Connected to MQTT broker\")\n",
    "            self.connected = True\n",
    "            \n",
    "            # Resubscribe to topics\n",
    "            for topic in self.subscriptions:\n",
    "                client.subscribe(topic)\n",
    "        else:\n",
    "            print(f\"Failed to connect to MQTT broker with code {rc}\")\n",
    "    \n",
    "    def _on_message(self, client, userdata, msg):\n",
    "        try:\n",
    "            payload = json.loads(msg.payload.decode())\n",
    "            \n",
    "            # Call the appropriate callback\n",
    "            for topic, callback in self.subscriptions.items():\n",
    "                if mqtt.topic_matches_sub(topic, msg.topic):\n",
    "                    callback(msg.topic, payload)\n",
    "        except json.JSONDecodeError:\n",
    "            print(f\"Received invalid JSON on {msg.topic}\")\n",
    "    \n",
    "    def _on_disconnect(self, client, userdata, rc):\n",
    "        print(\"Disconnected from MQTT broker\")\n",
    "        self.connected = False"

## 3. AI-Powered Device Controller

In [ ]:
class AIDeviceController:\n",
    "    def __init__(self, client_id: str, broker: str = \"localhost\"):\n",
    "        self.mqtt = MQTTClient(client_id, broker)\n",
    "        self.devices: Dict[str, DeviceState] = {}\n",
    "        self.rules: List[Dict] = []\n",
    "    \n",
    "    def start(self):\n",
    "        \"\"\"Start the device controller.\"\"\"\n",
    "        # Connect to MQTT broker\n",
    "        self.mqtt.connect()\n",
    "        \n",
    "        # Subscribe to device status updates\n",
    "        self.mqtt.subscribe(\"devices/+/status\", self._handle_device_update)\n",
    "        \n",
    "    def stop(self):\n",
    "        \"\"\"Stop the device controller.\"\"\"\n",
    "        self.mqtt.disconnect()\n",
    "    \n",
    "    def register_device(self, device_id: str, device_type: DeviceType):\n",
    "        \"\"\"Register a new device with the controller.\"\"\"\n",
    "        if device_id not in self.devices:\n",
    "            self.devices[device_id] = DeviceState(device_id, device_type)\n",
    "            print(f\"Registered device {device_id} of type {device_type.value}\")\n",
    "            return True\n",
    "        else:\n",
    "            print(f\"Device {device_id} already registered\")\n",
    "            return False\n",
    "    \n",
    "    def add_rule(self, rule: Dict):\n",
    "        \"\"\"Add a rule for device control.\"\"\"\n",
    "        self.rules.append(rule)\n",
    "    \n",
    "    def send_command(self, device_id: str, command: str, parameters: Dict = None):\n",
    "        \"\"\"Send a command to a device.\"\"\"\n",
    "        if device_id not in self.devices:\n",
    "            print(f\"Device {device_id} not registered\")\n",
    "            return False\n",
    "        \n",
    "        message = {\n",
    "            \"command\": command,\n",
    "            \"timestamp\": time.time()\n",
    "        }\n",
    "        \n",
    "        if parameters:\n",
    "            message[\"parameters\"] = parameters\n",
    "        \n",
    "        return self.mqtt.publish(f\"devices/{device_id}/commands\", message)\n",
    "    \n",
    "    def get_device_status(self, device_id: str) -> Optional[Dict]:\n",
    "        \"\"\"Get the status of a device.\"\"\"\n",
    "        if device_id in self.devices:\n",
    "            return self.devices[device_id].to_dict()\n",
    "        return None\n",
    "    \n",
    "    def _handle_device_update(self, topic: str, payload: Dict):\n",
    "        \"\"\"Handle device status update messages.\"\"\"\n",
    "        # Extract device_id from topic (format: devices/{device_id}/status)\n",
    "        parts = topic.split('/')\n",
    "        if len(parts) == 3 and parts[0] == \"devices\" and parts[2] == \"status\":\n",
    "            device_id = parts[1]\n",
    "            \n",
    "            if device_id in self.devices:\n",
    "                # Update existing device\n",
    "                self.devices[device_id].update(payload)\n",
    "                print(f\"Updated device {device_id} status: {self.devices[device_id].status}\")\n",
    "                \n",
    "                # Check rules\n",
    "                self._evaluate_rules(device_id)\n",
    "            else:\n",
    "                # Auto-register new device if type is provided\n",
    "                if \"device_type\" in payload:\n",
    "                    try:\n",
    "                        device_type = DeviceType(payload[\"device_type\"])\n",
    "                        self.register_device(device_id, device_type)\n",
    "                        self.devices[device_id].update(payload)\n",
    "                    except ValueError:\n",
    "                        print(f\"Invalid device type: {payload.get('device_type')}\")\n",
    "    \n",
    "    def _evaluate_rules(self, device_id: str):\n",
    "        \"\"\"Evaluate rules for a device and take actions.\"\"\"\n",
    "        if device_id not in self.devices:\n",
    "            return\n",
    "        \n",
    "        device = self.devices[device_id]\n",
    "        \n",
    "        for rule in self.rules:\n",
    "            if self._rule_matches(rule, device):\n",
    "                self._execute_action(rule[\"action\"], device_id)"
   

In [ ]:
# Example usage\n",
    "def main():\n",
    "    # Create AI controller\n",
    "    controller = AIDeviceController(\"ai-controller\")\n",
    "    controller.start()\n",
    "    \n",
    "    # Register devices\n",
    "    controller.register_device(\"qpu-001\", DeviceType.QUANTUM_PROCESSOR)\n",
    "    controller.register_device(\"cryo-001\", DeviceType.CRYOGENIC_SYSTEM)\n",
    "    \n",
    "    # Add a rule\n",
    "    controller.add_rule({\n",
    "        \"condition\": {\n",
    "            \"device_type\": DeviceType.CRYOGENIC_SYSTEM.value,\n",
    "            \"parameters\": {\n",
    "                \"temperature\": {\"above\": 4.0}  # Temperature above 4K\n",
    "            }\n",
    "        },\n",
    "        \"action\": {\n",
    "            \"type\": \"command\",\n",
    "            \"target\": \"qpu-001\",\n",
    "            \"command\": \"shutdown\",\n",
    "            \"reason\": \"Temperature too high\"\n",
    "        }\n",
    "    })\n",
    "    \n",
    "    # Simulate a temperature increase\n",
    "    controller.mqtt.publish(\"devices/cryo-001/status\", {\n",
    "        \"status\": \"online\",\n",
    "        \"parameters\": {\n",
    "            \"temperature\": 4.5,\n",
    "            \"pressure\": 1.0\n",
    "        }\n",
    "    })\n",
    "    \n",
    "    # Wait for rules to be processed\n",
    "    time.sleep(1)\n",
    "    \n",
    "    # Check device status\n",
    "    print(controller.get_device_status(\"qpu-001\"))\n",
    "    print(controller.get_device_status(\"cryo-001\"))\n",
    "    \n",
    "    # Stop controller\n",
    "    controller.stop()\n",
    "\n",
    "# In a real environment, you would call main()\n",
    "# For the tutorial, we'll just display the code"
   ]

"metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.10"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}